# ICU Multimodal Encoders — Text (ClinicalBERT) + Image (DenseNet-121)
> **Text source** : MIMIC-IV-Note `mimiciv_note.discharge` via BigQuery  
> **Image source** : MIMIC-CXR-JPG v2.1 via GCS + BigQuery metadata  
> **Text output** : 768-dim ClinicalBERT embeddings per stay  
> **Image output** : 1024-dim DenseNet features + 14 CheXpert label probs per stay  
> **Leakage policy** : only notes/CXR from within ICU window; same train/val/test split as encoder notebook


In [1]:
#!pip install -q "Pillow==9.5.0" && import importlib, PIL; importlib.reload(PIL)

## Step 0 — Install Dependencies

In [2]:
import subprocess, sys

# Step 1: Restore Pillow to the system version (11.3.0) — don't fight it
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--force-reinstall', 'Pillow==11.3.0'],
               capture_output=True)

# Step 2: Install everything else (no Pillow upgrade allowed)
PACKAGES = [
    'torch', 'torchvision', 'transformers',
    'google-cloud-bigquery', 'google-cloud-storage',
    'db-dtypes', 'umap-learn', 'tqdm',
]
for pkg in PACKAGES:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True, text=True)
    print(f'  [{"OK" if r.returncode==0 else "FAIL"}] {pkg}')

# Step 3: Install torchxrayvision from source (latest main has Pillow 11 fix)
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'git+https://github.com/mlmed/torchxrayvision.git'],
    capture_output=True, text=True
)
print(f'  [{"OK" if r.returncode==0 else "FAIL"}] torchxrayvision (from source)')
if r.returncode != 0:
    print(r.stderr[-500:])

# Step 4: Patch the broken _typing.py in place so _Ink exists
patch = """
from typing import Union, Tuple
# Pillow 11.x compatibility patch
try:
    from ._typing import _Ink
except ImportError:
    _Ink = Union[
        Tuple[int, int, int, int],
        Tuple[int, int, int],
        Tuple[int, int],
        int,
        float,
        str,
    ]
"""
import pathlib, PIL
typing_file = pathlib.Path(PIL.__file__).parent / '_typing.py'
content = typing_file.read_text()
if '_Ink' not in content:
    with open(typing_file, 'a') as f:
        f.write('\n_Ink = Union[int, float, str, tuple]\n')
    print('✅ Patched PIL/_typing.py with _Ink')
else:
    print('✅ PIL/_typing.py already has _Ink')

# Step 5: Clear PIL from sys.modules and reimport fresh
import sys
for mod in list(sys.modules.keys()):
    if 'PIL' in mod:
        del sys.modules[mod]

# Step 6: Verify
import PIL, torch, transformers, torchxrayvision as xrv
from PIL import Image
print(f'\nPillow       : {PIL.__version__}')
print(f'PyTorch      : {torch.__version__}')
print(f'Transformers : {transformers.__version__}')
print(f'TXV (CXR)    : {xrv.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device   : {torch.cuda.get_device_name(0)}')
print('✅ All packages ready.')

  [OK] torch
  [OK] torchvision
  [OK] transformers
  [OK] google-cloud-bigquery
  [OK] google-cloud-storage
  [OK] db-dtypes
  [OK] umap-learn
  [OK] tqdm
  [OK] torchxrayvision (from source)
✅ Patched PIL/_typing.py with _Ink

Pillow       : 11.3.0
PyTorch      : 2.10.0+cu128
Transformers : 5.0.0
TXV (CXR)    : 1.4.0
GPU available: True
GPU device   : NVIDIA RTX PRO 6000 Blackwell Server Edition
✅ All packages ready.


## Step 1 — Environment Setup & Paths

In [3]:
import warnings, logging, os
from pathlib import Path
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('ICU_MultiEncoder')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm

from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()   # authenticates gcloud for BigQuery + GCS

# ── GCP project — must match the one with PhysioNet BQ access
GCP_PROJECT  = 'icuaiassistanttextencoder'   # <-- CHANGE THIS to your project
BQ_DATASET_NOTE = 'physionet-data.mimiciv_note'
BQ_DATASET_CXR  = 'physionet-data.mimic_cxr'
GCS_CXR_BUCKET  = 'mimic-cxr-jpg'       # public PhysioNet GCS bucket

# ── Local paths
BASE      = Path('/content/drive/MyDrive/Multimoal_ICUAIAssistant_Claud/Parquet')
MODEL_DIR = Path('/content/drive/MyDrive/Multimoal_ICUAIAssistant_Claud/Models')
CXR_CACHE = Path('/content/cxr_cache')   # ephemeral Colab disk — fast I/O
CXR_CACHE.mkdir(exist_ok=True)

TEXT_EMB_OUT = BASE / 'icu_text_embeddings.parquet'
CXR_EMB_OUT  = BASE / 'icu_cxr_embeddings.parquet'
TEXT_RPT_PNG = Path('/content/drive/MyDrive/Multimoal_ICUAIAssistant_Claud/icu_text_encoder_report.png')
CXR_RPT_PNG  = Path('/content/drive/MyDrive/Multimoal_ICUAIAssistant_Claud/icu_cxr_encoder_report.png')

DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
log.info('Device: %s', DEVICE)

# ── Plot theme
DARK_BG='#0F172A'; CARD_BG='#1E293B'; TEXT='#F1F5F9'; GRID='#334155'
PAL={'blue':'#2563EB','teal':'#0891B2','purple':'#7C3AED',
     'green':'#16A34A','orange':'#EA580C','red':'#DC2626','amber':'#CA8A04'}
plt.rcParams.update({
    'figure.facecolor':DARK_BG,'axes.facecolor':CARD_BG,'axes.edgecolor':GRID,
    'axes.labelcolor':TEXT,'axes.titlecolor':TEXT,'xtick.color':TEXT,
    'ytick.color':TEXT,'text.color':TEXT,'grid.color':GRID,'grid.linewidth':0.5,
    'axes.grid':True,'axes.titlesize':12,'axes.labelsize':10,
    'xtick.labelsize':8,'ytick.labelsize':8,'legend.fontsize':8,
})
log.info('Environment ready.')


Mounted at /content/drive


## Step 2 — Load ICU Stay Cohort & Split Labels
> Reuse the exact same train/val/test split from the encoder notebook.


In [4]:
static  = pd.read_parquet(BASE / 'icu_static_features.parquet')
encoder_feats = pd.read_parquet(BASE / 'icu_encoder_features.parquet')

# stay cohort: stay_id, subject_id, hadm_id, intime, outtime, split
cohort = static[['stay_id','subject_id','hadm_id','intime','outtime']].copy()
cohort['intime']  = pd.to_datetime(cohort['intime'])
cohort['outtime'] = pd.to_datetime(cohort['outtime'])

# Attach split labels from encoder features parquet
split_map = encoder_feats[['stay_id','split']]
cohort = cohort.merge(split_map, on='stay_id', how='left')
cohort['split'] = cohort['split'].fillna('train')  # default if not matched

log.info('Cohort: %d stays | %d unique subjects',
         len(cohort), cohort['subject_id'].nunique())
log.info('Split distribution:\n%s', cohort['split'].value_counts().to_string())

# Unique hadm_ids for BigQuery filtering
HADM_IDS   = cohort['hadm_id'].dropna().astype(int).unique().tolist()
SUBJECT_IDS = cohort['subject_id'].dropna().astype(int).unique().tolist()
log.info('Unique hadm_ids for BQ query: %d', len(HADM_IDS))


## Step 3 — Pull Clinical Notes via BigQuery
> Discharge summaries only — linked to ICU stays via hadm_id.  
> Temporal filter: note charttime must be <= ICU outtime (no future leakage).


In [5]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=GCP_PROJECT)

# ── Build hadm_id filter string in batches (BQ IN clause limit ~10k)
def bq_query_notes(hadm_ids: list, project: str) -> pd.DataFrame:
    """Pull discharge notes for the cohort. Batched to avoid BQ limits."""
    BATCH = 5000
    all_dfs = []
    for i in range(0, len(hadm_ids), BATCH):
        batch = hadm_ids[i:i+BATCH]
        ids_str = ','.join(str(x) for x in batch)
        query = f"""
            SELECT
                n.subject_id,
                n.hadm_id,
                n.note_id,
                n.charttime,
                n.note_type,
                n.text
            FROM `{project}.mimiciv_note.discharge` AS n
            WHERE n.hadm_id IN ({ids_str})
              AND n.text IS NOT NULL
              AND LENGTH(n.text) > 100
        """
        df = bq_client.query(query).to_dataframe()
        all_dfs.append(df)
        log.info('BQ note batch %d/%d: %d rows',
                 i//BATCH+1, (len(hadm_ids)-1)//BATCH+1, len(df))

    return pd.concat(all_dfs, ignore_index=True)

log.info('Pulling discharge notes from BigQuery...')
notes_raw = bq_query_notes(HADM_IDS, BQ_DATASET_NOTE.split('.')[0])
notes_raw['charttime'] = pd.to_datetime(notes_raw['charttime'], utc=True)
log.info('Raw notes pulled: %d rows', len(notes_raw))

# ── Temporal leakage filter: drop notes after ICU discharge
notes_raw = notes_raw.merge(
    cohort[['hadm_id','intime','outtime']], on='hadm_id', how='inner'
)
notes_raw['outtime_utc'] = pd.to_datetime(notes_raw['outtime'], utc=True)
notes_raw['intime_utc']  = pd.to_datetime(notes_raw['intime'],  utc=True)

# Keep only notes written during or before ICU stay
notes_clean = notes_raw[
    notes_raw['charttime'] <= notes_raw['outtime_utc']
].copy()

# For hadm with no charttime (some discharge notes have NULL), keep them
notes_null_time = notes_raw[notes_raw['charttime'].isna()].copy()
notes_clean = pd.concat([notes_clean, notes_null_time], ignore_index=True)

log.info('Notes after temporal filter: %d (removed %d post-discharge)',
         len(notes_clean), len(notes_raw) - len(notes_clean))
log.info('Note type distribution:\n%s',
         notes_clean['note_type'].value_counts().head(10).to_string())


## Step 4 — ClinicalBERT Text Encoder
> Model: `emilyalsentzer/Bio_ClinicalBERT` — pretrained on MIMIC-III clinical notes.  
> Strategy: mean-pool last 4 hidden layers → 768-dim embedding per note.  
> Aggregation: mean of all note embeddings per stay (most stays have 1–3 notes).


In [6]:
import torch
from transformers import AutoTokenizer, AutoModel

BERT_MODEL = 'emilyalsentzer/Bio_ClinicalBERT'
MAX_TOKENS = 512   # BERT hard limit
BATCH_SIZE = 16    # notes per forward pass — reduce to 8 if OOM on CPU

log.info('Loading ClinicalBERT tokenizer and model...')
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model = AutoModel.from_pretrained(BERT_MODEL, output_hidden_states=True)
bert_model.eval()
bert_model.to(DEVICE)
log.info('ClinicalBERT loaded on %s', DEVICE)


def clean_note_text(text: str) -> str:
    """Remove PHI placeholders and normalise whitespace."""
    import re
    text = re.sub(r'\[\*\*.*?\*\*\]', ' ', text)  # remove [**PHI**] markers
    text = re.sub(r'\s+', ' ', text).strip()
    return text


@torch.no_grad()
def embed_texts(texts: list[str]) -> np.ndarray:
    """
    Returns (N, 768) array of ClinicalBERT embeddings.
    Strategy: mean-pool last 4 hidden layers → then mean across token positions.
    This captures richer semantics than CLS alone.
    """
    all_embeddings = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i+BATCH_SIZE]
        batch = [clean_note_text(t)[:2000] for t in batch]  # pre-truncate string
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_TOKENS,
            return_tensors='pt'
        )
        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

        outputs = bert_model(**encoded)
        hidden_states = outputs.hidden_states  # tuple of 13 layers

        # Mean-pool last 4 hidden layers
        last4 = torch.stack(hidden_states[-4:], dim=0)  # (4, B, T, 768)
        token_embeddings = last4.mean(dim=0)            # (B, T, 768)

        # Mean across non-padding token positions
        attention_mask = encoded['attention_mask'].unsqueeze(-1).float()
        summed = (token_embeddings * attention_mask).sum(dim=1)
        counts = attention_mask.sum(dim=1).clamp(min=1e-9)
        embeddings = (summed / counts).cpu().numpy()    # (B, 768)
        all_embeddings.append(embeddings)

    return np.vstack(all_embeddings)


# ── Encode all notes in batches
log.info('Encoding %d notes with ClinicalBERT...', len(notes_clean))
texts = notes_clean['text'].fillna('').tolist()
note_embeddings = embed_texts(texts)
log.info('Note embeddings shape: %s', note_embeddings.shape)

# Attach embeddings back to notes dataframe
emb_cols = [f'bert_dim_{i}' for i in range(768)]
emb_df = pd.DataFrame(note_embeddings, columns=emb_cols)
notes_with_emb = pd.concat(
    [notes_clean[['hadm_id','note_id','note_type']].reset_index(drop=True), emb_df],
    axis=1
)


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

## Step 5 — Aggregate Note Embeddings to Stay Level
> Multiple notes per hadm → mean embedding vector. One row per stay_id.


In [7]:
# Mean across all notes for the same admission
stay_text_emb = (
    notes_with_emb
    .groupby('hadm_id')[emb_cols]
    .mean()
    .reset_index()
)

# Join back to stay_id via cohort
stay_text_emb = cohort[['stay_id','hadm_id','split']].merge(
    stay_text_emb, on='hadm_id', how='left'
)

# Coverage: how many stays have at least one note?
has_note = stay_text_emb[emb_cols[0]].notna().sum()
total    = len(stay_text_emb)
log.info('Text embedding coverage: %d/%d stays (%.1f%%)',
         has_note, total, has_note/total*100)

# Stays without notes get zero vector (marked with flag)
stay_text_emb['text_available'] = stay_text_emb[emb_cols[0]].notna().astype(int)
stay_text_emb[emb_cols] = stay_text_emb[emb_cols].fillna(0.0)

log.info('Text embedding table: %s', stay_text_emb.shape)
stay_text_emb[['stay_id','hadm_id','split','text_available']].head()


,stay_id,hadm_id,split,text_available
0,39553978,29079034,train,0
1,37081114,25860671,val,0
2,39765666,26913865,test,0
3,37067082,24597018,train,0
4,34592300,27703517,train,0


## Step 6 — Pull CXR Metadata via BigQuery
> Frontal views only (PA/AP). Most recent CXR within ±48h of ICU admission.


In [14]:
import requests
import getpass
from requests.auth import HTTPBasicAuth

print("Enter your PhysioNet credentials:")
PHYSIONET_USER = input("Username: ")
PHYSIONET_PASS = getpass.getpass("Password: ")

AUTH = HTTPBasicAuth(PHYSIONET_USER, PHYSIONET_PASS)

# ── Use the actual data file as the auth probe (kills two birds)
AUTH_CHECK_URL = "https://physionet.org/content/mimic-cxr-jpg/2.1.0/"
test = requests.get(AUTH_CHECK_URL, auth=AUTH)

if test.status_code == 200:
    log.info("PhysioNet auth OK ✅")
elif test.status_code == 401:
    raise ValueError("Wrong username or password")
elif test.status_code == 403:
    raise ValueError("Credentials OK but MIMIC-CXR-JPG DUA not signed — "
                     "go to physionet.org/content/mimic-cxr-jpg/ and sign it")
elif test.status_code == 404:
    raise ValueError("URL not found — check PhysioNet is reachable")
else:
    raise ValueError(f"Unexpected response: {test.status_code}")

Enter your PhysioNet credentials:
Username: gomathimeena.m@gmail.com
Password: ··········


In [10]:
def bq_query_cxr(subject_ids: list, project: str) -> pd.DataFrame:
    """Pull frontal CXR metadata from MIMIC-CXR-JPG BigQuery."""
    BATCH = 5000
    all_dfs = []
    for i in range(0, len(subject_ids), BATCH):
        batch = subject_ids[i:i+BATCH]
        ids_str = ','.join(str(x) for x in batch)
        # mimic_cxr.record_list links subject_id → study_id → dicom_id + path
        query = f"""
            SELECT
                r.subject_id,
                r.study_id,
                r.dicom_id,
                r.path,
                m.StudyDate,
                m.ViewPosition
            FROM `physionet-data.mimic_cxr.record_list` AS r
            LEFT JOIN `physionet-data.mimic_cxr.dicom_metadata_string` AS m
                ON r.dicom_id = m.SOPInstanceUID
            WHERE r.subject_id IN ({ids_str})
              AND m.ViewPosition IN ('PA', 'AP')
        """
        df = bq_client.query(query).to_dataframe()
        all_dfs.append(df)
        log.info('BQ CXR batch %d/%d: %d rows',
                 i//BATCH+1, (len(subject_ids)-1)//BATCH+1, len(df))

    return pd.concat(all_dfs, ignore_index=True)


log.info('Pulling CXR metadata from BigQuery...')
cxr_meta = bq_query_cxr(SUBJECT_IDS, GCP_PROJECT)
log.info('CXR metadata rows: %d', len(cxr_meta))

# Parse StudyDate to datetime
cxr_meta['study_dt'] = pd.to_datetime(
    cxr_meta['StudyDate'], format='%Y%m%d', errors='coerce'
)

# ── Temporal filter: CXR within 48h before or during ICU admission
cxr_meta = cxr_meta.merge(
    cohort[['subject_id','stay_id','intime','outtime','split']],
    on='subject_id', how='inner'
)
cxr_meta['intime']  = pd.to_datetime(cxr_meta['intime'])
cxr_meta['window_start'] = cxr_meta['intime'] - pd.Timedelta(hours=48)
cxr_meta['window_end']   = cxr_meta['intime'] + pd.Timedelta(hours=24)

cxr_filtered = cxr_meta[
    (cxr_meta['study_dt'] >= cxr_meta['window_start']) &
    (cxr_meta['study_dt'] <= cxr_meta['window_end'])
].copy()

# Keep most recent CXR per stay
cxr_filtered = (
    cxr_filtered
    .sort_values(['stay_id','study_dt'], ascending=[True, False])
    .drop_duplicates(subset='stay_id', keep='first')
    .reset_index(drop=True)
)

log.info('Stays with valid CXR: %d / %d (%.1f%%)',
         len(cxr_filtered), len(cohort), len(cxr_filtered)/len(cohort)*100)
log.info('View distribution:\n%s',
         cxr_filtered['ViewPosition'].value_counts().to_string())


Forbidden: 403 Access Denied: Table physionet-data:mimic_cxr.dicom_metadata_string: User does not have permission to query table physionet-data:mimic_cxr.dicom_metadata_string, or perhaps it does not exist.; reason: accessDenied, message: Access Denied: Table physionet-data:mimic_cxr.dicom_metadata_string: User does not have permission to query table physionet-data:mimic_cxr.dicom_metadata_string, or perhaps it does not exist.

Location: US
Job ID: e5937813-83c5-42a4-8f59-a58e53cdd211


## Step 7 — DenseNet-121 CXR Encoder (torchxrayvision)
> Model pretrained on CheXpert + NIH + PadChest datasets via `torchxrayvision`.  
> Output: 1024-dim feature vector + 14 pathology label probabilities per image.  
> Images downloaded from GCS on demand and cached to Colab disk.


In [ ]:
import torchxrayvision as xrv
import torchvision.transforms as transforms
from PIL import Image
from google.cloud import storage as gcs
import io

# ── Load DenseNet-121 pretrained on all datasets
log.info('Loading DenseNet-121 (torchxrayvision)...')
cxr_model = xrv.models.DenseNet(weights='densenet121-res224-all')
cxr_model.eval()
cxr_model.to(DEVICE)

# 14 CheXpert/NIH pathology labels
PATHOLOGY_LABELS = cxr_model.pathologies  # list of 18 strings
log.info('Pathology labels: %s', PATHOLOGY_LABELS)

# ── GCS client for image download
gcs_client = gcs.Client(project=GCP_PROJECT)
gcs_bucket = gcs_client.bucket(GCS_CXR_BUCKET)


def load_cxr_from_gcs(gcs_path: str) -> np.ndarray | None:
    """
    Download a JPEG from GCS and convert to normalised numpy array.
    torchxrayvision expects values in [-1024, 1024] range.
    """
    # path format in BQ: files/p10/p10000032/s50414267/02aa804e-bde0afdd.jpg
    blob = gcs_bucket.blob(gcs_path)
    try:
        img_bytes = blob.download_as_bytes()
        img = Image.open(io.BytesIO(img_bytes)).convert('L')  # grayscale
        img_arr = np.array(img).astype(np.float32)
        # Normalise to xrv expected range
        img_arr = xrv.datasets.normalize(img_arr, maxval=255, reshape=True)
        return img_arr
    except Exception as e:
        log.warning('Failed to load CXR %s: %s', gcs_path, e)
        return None


@torch.no_grad()
def encode_cxr_batch(img_arrays: list) -> tuple[np.ndarray, np.ndarray]:
    """
    Returns:
      features  : (N, 1024) penultimate layer embeddings
      label_probs: (N, len(PATHOLOGY_LABELS)) sigmoid probabilities
    """
    transform = transforms.Compose([
        xrv.datasets.XRayCenterCrop(),
        xrv.datasets.XRayResizer(224),
    ])

    tensors = []
    for arr in img_arrays:
        t = transform(arr)                              # (1, 224, 224)
        tensors.append(torch.tensor(t, dtype=torch.float32))

    batch = torch.stack(tensors).to(DEVICE)            # (B, 1, 224, 224)

    # Feature extraction: hook penultimate layer
    feature_vec = []
    def hook_fn(module, inp, out):
        pooled = out.mean(dim=[2,3])                   # global avg pool
        feature_vec.append(pooled.cpu().numpy())

    # Register hook on the last DenseBlock
    hook = cxr_model.features.denseblock4.register_forward_hook(hook_fn)

    logits = cxr_model(batch)                          # (B, n_labels)
    hook.remove()

    label_probs = torch.sigmoid(logits).cpu().numpy()  # (B, n_labels)
    features    = feature_vec[0]                       # (B, 1024)

    return features, label_probs


# ── Process all CXRs
CXR_BATCH = 8   # images per forward pass
all_feat_rows = []

paths    = cxr_filtered['path'].tolist()
stay_ids = cxr_filtered['stay_id'].tolist()
splits   = cxr_filtered['split'].tolist()

log.info('Encoding %d CXR images...', len(paths))

i = 0
with tqdm(total=len(paths), desc='CXR encoding') as pbar:
    while i < len(paths):
        batch_paths    = paths[i:i+CXR_BATCH]
        batch_stay_ids = stay_ids[i:i+CXR_BATCH]
        batch_splits   = splits[i:i+CXR_BATCH]

        imgs = [load_cxr_from_gcs(p) for p in batch_paths]
        valid_mask = [img is not None for img in imgs]
        valid_imgs = [img for img in imgs if img is not None]

        if valid_imgs:
            feats, probs = encode_cxr_batch(valid_imgs)
            j = 0
            for k, (sid, sp, valid) in enumerate(
                    zip(batch_stay_ids, batch_splits, valid_mask)):
                if valid:
                    row = {'stay_id': sid, 'split': sp, 'cxr_available': 1}
                    for d in range(feats.shape[1]):
                        row[f'cxr_feat_{d}'] = float(feats[j, d])
                    for lbl, prob in zip(PATHOLOGY_LABELS, probs[j]):
                        safe_lbl = lbl.replace(' ','_').replace('/','_')
                        row[f'cxr_prob_{safe_lbl}'] = float(prob)
                    all_feat_rows.append(row)
                    j += 1

        i += CXR_BATCH
        pbar.update(len(batch_paths))

cxr_emb_df = pd.DataFrame(all_feat_rows)
log.info('CXR embedding table: %s', cxr_emb_df.shape)


## Step 8 — Merge CXR Embeddings with Full Stay Cohort
> Stays without a CXR get zero vector + `cxr_available = 0`.


In [ ]:
FEAT_COLS_CXR = [c for c in cxr_emb_df.columns
                 if c.startswith('cxr_feat_')]
PROB_COLS_CXR = [c for c in cxr_emb_df.columns
                 if c.startswith('cxr_prob_')]

# Full cohort merge — left join so all stays are present
cxr_full = cohort[['stay_id','split']].merge(
    cxr_emb_df, on=['stay_id','split'], how='left'
)
cxr_full['cxr_available'] = cxr_full['cxr_available'].fillna(0).astype(int)
cxr_full[FEAT_COLS_CXR + PROB_COLS_CXR] = \
    cxr_full[FEAT_COLS_CXR + PROB_COLS_CXR].fillna(0.0)

cxr_coverage = cxr_full['cxr_available'].mean() * 100
log.info('CXR coverage: %.1f%% of all stays', cxr_coverage)
log.info('CXR embedding table (full cohort): %s', cxr_full.shape)

# Pathology label prevalence
prob_means = cxr_full[PROB_COLS_CXR].mean().sort_values(ascending=False)
print('\nTop pathology probabilities (mean across all stays with CXR):')
print(prob_means.to_string())


## Step 9 — Save Embeddings to Parquet

In [ ]:
# ── Text embeddings
stay_text_emb.to_parquet(TEXT_EMB_OUT, index=False,
                         engine='pyarrow', compression='snappy')
log.info('Text embeddings saved: %s | %s', TEXT_EMB_OUT, stay_text_emb.shape)

# ── CXR embeddings + label probs
cxr_full.to_parquet(CXR_EMB_OUT, index=False,
                    engine='pyarrow', compression='snappy')
log.info('CXR embeddings saved: %s | %s', CXR_EMB_OUT, cxr_full.shape)

# ── Save column name metadata for fusion layer
import json
emb_meta = {
    'text_emb_cols':  emb_cols,
    'text_dim':       768,
    'cxr_feat_cols':  FEAT_COLS_CXR,
    'cxr_prob_cols':  PROB_COLS_CXR,
    'cxr_feat_dim':   len(FEAT_COLS_CXR),
    'pathology_labels': [str(l) for l in PATHOLOGY_LABELS],
}
with open(MODEL_DIR / 'embedding_meta.json','w') as f:
    json.dump(emb_meta, f, indent=2)
log.info('Embedding metadata saved.')


## Step 10 — Text Encoder Report Dashboard

In [ ]:
from umap import UMAP
from sklearn.preprocessing import LabelEncoder

scores = pd.read_parquet(BASE / 'icu_risk_scores.parquet')
TIER_COLORS = {'LOW':'#16A34A','MODERATE':'#CA8A04','HIGH':'#EA580C',
               'SEVERE':'#DC2626','CRITICAL':'#7C3AED'}
TIER_ORDER  = ['LOW','MODERATE','HIGH','SEVERE','CRITICAL']

fig = plt.figure(figsize=(22, 20), facecolor=DARK_BG)
fig.suptitle('ClinicalBERT Text Encoder — Report Dashboard',
             fontsize=18, fontweight='bold', color=TEXT, y=0.99)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.52, wspace=0.38)
def sp(ax): ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# ── 1. UMAP of text embeddings coloured by criticality tier
ax1 = fig.add_subplot(gs[0, :2])
sample_idx = stay_text_emb[stay_text_emb['text_available']==1].sample(
    min(3000, stay_text_emb['text_available'].sum()), random_state=42).index
emb_sample = stay_text_emb.loc[sample_idx, emb_cols].values
sid_sample  = stay_text_emb.loc[sample_idx, 'stay_id'].values

umap_model = UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42)
umap_2d = umap_model.fit_transform(emb_sample)

tier_map = scores.set_index('stay_id')['criticality_tier']
tiers = pd.Series(sid_sample).map(tier_map).fillna('LOW')

for tier in TIER_ORDER:
    mask = tiers == tier
    ax1.scatter(umap_2d[mask,0], umap_2d[mask,1],
                c=TIER_COLORS[tier], s=6, alpha=0.5, label=tier, edgecolors='none')
ax1.set_title('UMAP of ClinicalBERT Embeddings (coloured by criticality tier)')
ax1.set_xlabel('UMAP dim 1'); ax1.set_ylabel('UMAP dim 2')
ax1.legend(markerscale=3, framealpha=0.2, facecolor=CARD_BG); sp(ax1)

# ── 2. Note coverage by split
ax2 = fig.add_subplot(gs[0, 2])
cov = stay_text_emb.groupby('split')['text_available'].agg(['sum','count'])
cov['pct'] = cov['sum'] / cov['count'] * 100
cov = cov.reindex(['train','val','test'])
bars = ax2.bar(cov.index, cov['pct'],
               color=[PAL['blue'],PAL['teal'],PAL['purple']],
               edgecolor=DARK_BG, width=0.5)
for bar, val in zip(bars, cov['pct']):
    ax2.text(bar.get_x()+bar.get_width()/2, val+0.5,
             f'{val:.1f}%', ha='center', fontsize=9, color=TEXT)
ax2.set_title('Note Coverage by Split'); ax2.set_ylabel('% stays with note')
ax2.set_ylim(0,105); sp(ax2)

# ── 3. Notes per admission histogram
ax3 = fig.add_subplot(gs[1, 0])
notes_per_hadm = notes_clean.groupby('hadm_id').size()
ax3.hist(notes_per_hadm.clip(upper=10), bins=range(1,12),
         color=PAL['teal'], edgecolor=DARK_BG, width=0.8)
ax3.set_title('Notes per Admission')
ax3.set_xlabel('Number of notes'); ax3.set_ylabel('Admissions'); sp(ax3)

# ── 4. Note length distribution
ax4 = fig.add_subplot(gs[1, 1])
note_lens = notes_clean['text'].str.len().clip(upper=5000)
ax4.hist(note_lens, bins=50, color=PAL['purple'], edgecolor=DARK_BG)
ax4.axvline(note_lens.median(), color='white', ls='--', lw=1.5,
            label=f'Median {note_lens.median():.0f} chars')
ax4.set_title('Note Length Distribution')
ax4.set_xlabel('Characters (capped 5000)'); ax4.set_ylabel('Count')
ax4.legend(framealpha=0.2, facecolor=CARD_BG); sp(ax4)

# ── 5. Embedding L2-norm distribution by tier
ax5 = fig.add_subplot(gs[1, 2])
norms = np.linalg.norm(stay_text_emb[emb_cols].values, axis=1)
stay_text_emb['emb_norm'] = norms
merge_t = stay_text_emb.merge(scores[['stay_id','criticality_tier']], on='stay_id', how='left')
for tier in TIER_ORDER:
    d = merge_t.loc[merge_t['criticality_tier']==tier,'emb_norm']
    if len(d)>5: ax5.hist(d, bins=30, alpha=0.6,
                         color=TIER_COLORS[tier], label=tier, edgecolor='none')
ax5.set_title('Embedding Norm by Tier')
ax5.set_xlabel('L2 norm'); ax5.set_ylabel('Stays')
ax5.legend(framealpha=0.2, facecolor=CARD_BG); sp(ax5)

# ── 6. Note type distribution
ax6 = fig.add_subplot(gs[2, :])
type_counts = notes_clean['note_type'].value_counts().head(10)
bars = ax6.barh(range(len(type_counts)), type_counts.values,
                color=PAL['blue'], edgecolor=DARK_BG, height=0.6)
ax6.set_yticks(range(len(type_counts)))
ax6.set_yticklabels(type_counts.index)
for bar, val in zip(bars, type_counts.values):
    ax6.text(val+50, bar.get_y()+bar.get_height()/2,
             f'{val:,}', va='center', fontsize=8, color=TEXT)
ax6.set_title('Clinical Note Types in Cohort')
ax6.set_xlabel('Number of notes'); sp(ax6)

plt.savefig(TEXT_RPT_PNG, dpi=180, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
log.info('Text encoder report saved.')


## Step 11 — CXR Encoder Report Dashboard

In [ ]:
fig2 = plt.figure(figsize=(22, 20), facecolor=DARK_BG)
fig2.suptitle('DenseNet-121 CXR Encoder — Report Dashboard',
              fontsize=18, fontweight='bold', color=TEXT, y=0.99)
gs2 = gridspec.GridSpec(3, 3, figure=fig2, hspace=0.52, wspace=0.38)

# ── 1. UMAP of CXR features by tier
ax1 = fig2.add_subplot(gs2[0, :2])
cxr_has = cxr_full[cxr_full['cxr_available']==1]
sample_cxr = cxr_has.sample(min(2000, len(cxr_has)), random_state=42)
feat_sample = sample_cxr[FEAT_COLS_CXR].values

umap_cxr = UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42)
umap_cxr_2d = umap_cxr.fit_transform(feat_sample)

tiers_cxr = sample_cxr['stay_id'].map(tier_map).fillna('LOW')
for tier in TIER_ORDER:
    mask = tiers_cxr == tier
    ax1.scatter(umap_cxr_2d[mask,0], umap_cxr_2d[mask,1],
                c=TIER_COLORS[tier], s=6, alpha=0.5, label=tier, edgecolors='none')
ax1.set_title('UMAP of DenseNet-121 CXR Features (coloured by criticality tier)')
ax1.set_xlabel('UMAP dim 1'); ax1.set_ylabel('UMAP dim 2')
ax1.legend(markerscale=3, framealpha=0.2, facecolor=CARD_BG); sp(ax1)

# ── 2. CXR coverage by split
ax2 = fig2.add_subplot(gs2[0, 2])
cxr_cov = cxr_full.groupby('split')['cxr_available'].agg(['sum','count'])
cxr_cov['pct'] = cxr_cov['sum'] / cxr_cov['count'] * 100
cxr_cov = cxr_cov.reindex(['train','val','test'])
bars2 = ax2.bar(cxr_cov.index, cxr_cov['pct'],
                color=[PAL['orange'],PAL['teal'],PAL['purple']],
                edgecolor=DARK_BG, width=0.5)
for bar, val in zip(bars2, cxr_cov['pct']):
    ax2.text(bar.get_x()+bar.get_width()/2, val+0.5,
             f'{val:.1f}%', ha='center', fontsize=9, color=TEXT)
ax2.set_title('CXR Coverage by Split'); ax2.set_ylabel('% stays with CXR')
ax2.set_ylim(0, 105); sp(ax2)

# ── 3. Pathology label heatmap by criticality tier
ax3 = fig2.add_subplot(gs2[1, :])
cxr_with_tier = cxr_full.merge(
    scores[['stay_id','criticality_tier']], on='stay_id', how='left'
)
hmap_data = (
    cxr_with_tier.groupby('criticality_tier')[PROB_COLS_CXR].mean()
).reindex(TIER_ORDER)
clean_labels = [c.replace('cxr_prob_','').replace('_',' ') for c in PROB_COLS_CXR]
im = ax3.imshow(hmap_data.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=0.4)
ax3.set_xticks(range(len(PROB_COLS_CXR)))
ax3.set_xticklabels(clean_labels, rotation=35, ha='right', fontsize=8)
ax3.set_yticks(range(len(TIER_ORDER)))
ax3.set_yticklabels(TIER_ORDER)
for i in range(len(TIER_ORDER)):
    for j in range(len(PROB_COLS_CXR)):
        val = hmap_data.values[i,j]
        if not np.isnan(val):
            ax3.text(j, i, f'{val:.2f}', ha='center', va='center',
                     fontsize=7, color='white' if val>0.2 else TEXT)
plt.colorbar(im, ax=ax3, label='Mean probability', shrink=0.6)
ax3.set_title('Mean CXR Pathology Probabilities by Criticality Tier')

# ── 4. ViewPosition distribution
ax4 = fig2.add_subplot(gs2[2, 0])
vp_counts = cxr_filtered['ViewPosition'].value_counts()
ax4.pie(vp_counts, labels=vp_counts.index, autopct='%1.1f%%',
        colors=[PAL['blue'],PAL['teal']],
        wedgeprops=dict(width=0.5, edgecolor=DARK_BG, linewidth=2),
        textprops={'color':TEXT,'fontsize':9})
ax4.set_title('CXR View Position')

# ── 5. Days from admission to CXR
ax5 = fig2.add_subplot(gs2[2, 1])
cxr_filtered['days_to_cxr'] = (
    (cxr_filtered['study_dt'] - cxr_filtered['intime'])
    .dt.total_seconds() / 86400
)
ax5.hist(cxr_filtered['days_to_cxr'].clip(-2,2), bins=40,
         color=PAL['amber'], edgecolor=DARK_BG)
ax5.axvline(0, color='white', ls='--', lw=1, label='ICU admission')
ax5.set_title('CXR Timing Relative\nto ICU Admission')
ax5.set_xlabel('Days (negative=before)'); ax5.set_ylabel('CXR studies')
ax5.legend(framealpha=0.2, facecolor=CARD_BG); sp(ax5)

# ── 6. Top pathology prevalence bar chart
ax6 = fig2.add_subplot(gs2[2, 2])
path_prev = cxr_full.loc[
    cxr_full['cxr_available']==1, PROB_COLS_CXR
].apply(lambda col: (col > 0.5).mean() * 100).sort_values(ascending=False)
top_path = path_prev.head(10)
bars3 = ax6.barh(range(len(top_path)), top_path.values,
                 color=PAL['red'], edgecolor=DARK_BG, height=0.6)
ax6.set_yticks(range(len(top_path)))
ax6.set_yticklabels(
    [l.replace('cxr_prob_','').replace('_',' ') for l in top_path.index],
    fontsize=8)
for bar, val in zip(bars3, top_path.values):
    ax6.text(val+0.2, bar.get_y()+bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=8, color=TEXT)
ax6.set_title('Top Pathology Prevalence\n(prob > 0.5)')
ax6.set_xlabel('% of CXR stays'); ax6.invert_yaxis(); sp(ax6)

plt.savefig(CXR_RPT_PNG, dpi=180, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
log.info('CXR encoder report saved.')


---
# Explainability Layer
> **Text** : Which words/phrases in the clinical note drove the risk prediction — in plain language  
> **Image** : Which region of the chest X-ray the model focused on (Grad-CAM heatmap)  
> **Output** : Doctor-readable explanation card per patient


## Explainability A — ClinicalBERT Token Attention (Text)
> We extract the attention weights from BERT's last layer — high attention on a word means  
> the model leaned on that word when building the patient's embedding.  
> We then map those weights to plain clinical language a doctor can read.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re

# ── Clinical term → plain English dictionary
CLINICAL_PLAIN = {
    # Vitals
    'tachycardia':      'fast heart rate',
    'bradycardia':      'slow heart rate',
    'hypotension':      'low blood pressure',
    'hypertension':     'high blood pressure',
    'tachypnea':        'rapid breathing',
    'hypoxia':          'low blood oxygen',
    'febrile':          'fever',
    'afebrile':         'no fever',
    # Organ systems
    'aki':              'acute kidney injury',
    'arf':              'acute respiratory failure',
    'ards':             'severe lung failure',
    'sepsis':           'life-threatening infection response',
    'septic shock':     'infection causing dangerously low BP',
    'bacteremia':       'bacteria in the bloodstream',
    'pneumonia':        'lung infection',
    'intubated':        'breathing tube inserted',
    'intubation':       'placement of breathing tube',
    'ventilator':       'breathing machine',
    'mechanical ventilation': 'breathing machine support',
    'dialysis':         'kidney replacement therapy',
    'creatinine':       'kidney stress marker',
    'bilirubin':        'liver stress marker',
    'lactate':          'tissue oxygen deprivation marker',
    'troponin':         'heart muscle injury marker',
    'platelets':        'blood clotting cells',
    'wbc':              'white blood cell count (infection marker)',
    'altered mental status': 'confusion / reduced consciousness',
    'gcs':              'consciousness score',
    'pressors':         'blood pressure-raising medications',
    'vasopressors':     'medications to raise blood pressure',
    'norepinephrine':   'blood pressure medication (norepinephrine)',
    'dopamine':         'blood pressure medication (dopamine)',
    'dobutamine':       'heart support medication',
    'heparin':          'blood-thinning medication',
    'dnr':              'do-not-resuscitate order',
    'comfort measures': 'palliative / end-of-life care focus',
    'transfer':         'moved between care units',
    'icu':              'intensive care unit',
    'micu':             'medical intensive care unit',
    'sicu':             'surgical intensive care unit',
    'ccu':              'cardiac care unit',
    'copd':             'chronic lung disease',
    'chf':              'chronic heart failure',
    'cirrhosis':        'chronic liver scarring',
    'malignancy':       'cancer',
    'immunocompromised': 'weakened immune system',
    'obese':            'high body weight',
    'diabetes':         'blood sugar disorder',
    'coagulopathy':     'blood clotting disorder',
}

RISK_TERMS = {
    'high_risk': [
        'sepsis','septic shock','ards','intubated','vasopressors',
        'norepinephrine','aki','arf','altered mental status','dnr',
        'comfort measures','multi-organ','bacteremia','lactate','pressors'
    ],
    'moderate_risk': [
        'pneumonia','hypotension','tachycardia','hypoxia','ventilator',
        'dialysis','creatinine','troponin','gcs','transfer'
    ],
    'protective': [
        'afebrile','stable','improved','extubated','tolerating',
        'ambulating','discharged','weaned'
    ],
}


@torch.no_grad()
def get_token_attention(text: str) -> tuple[list[str], np.ndarray]:
    """
    Returns:
      tokens      : list of wordpiece tokens
      attn_weights: (n_tokens,) mean attention across all heads, last layer
    """
    text_clean = clean_note_text(text)[:2000]
    encoded = tokenizer(
        text_clean, return_tensors='pt',
        truncation=True, max_length=MAX_TOKENS,
        return_attention_mask=True
    )
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

    outputs = bert_model(**encoded, output_attentions=True)
    # Last layer attention: (1, n_heads, seq_len, seq_len)
    last_attn = outputs.attentions[-1][0]        # (n_heads, seq_len, seq_len)
    # Mean across heads, then mean of what each token attends TO
    token_attn = last_attn.mean(dim=0).mean(dim=0)  # (seq_len,)
    token_attn = token_attn.cpu().numpy()

    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0].cpu())
    return tokens, token_attn


def merge_wordpieces(tokens: list[str],
                     weights: np.ndarray) -> tuple[list[str], np.ndarray]:
    """Merge BERT ##wordpiece tokens back into whole words."""
    words, word_weights = [], []
    cur_word, cur_w = '', 0.0
    for tok, w in zip(tokens, weights):
        if tok in ('[CLS]','[SEP]','[PAD]'):
            continue
        if tok.startswith('##'):
            cur_word += tok[2:]
            cur_w = max(cur_w, w)   # take max weight of sub-tokens
        else:
            if cur_word:
                words.append(cur_word)
                word_weights.append(cur_w)
            cur_word, cur_w = tok, w
    if cur_word:
        words.append(cur_word)
        word_weights.append(cur_w)
    return words, np.array(word_weights)


def classify_word_risk(word: str) -> str:
    """Classify a word as high_risk / moderate_risk / protective / neutral."""
    w = word.lower()
    for category, terms in RISK_TERMS.items():
        if any(t in w or w in t for t in terms):
            return category
    return 'neutral'


def plain_name(word: str) -> str:
    """Return plain English name if known, else clean up the word."""
    w = word.lower()
    for term, plain in CLINICAL_PLAIN.items():
        if term in w:
            return plain
    return word


def explain_note_text(text: str, top_n: int = 8) -> dict:
    """
    Full plain-language explanation of what a clinical note is telling the model.
    Returns dict with: summary, high_risk_signals, protective_signals,
                       top_attended_words, attention_tokens (for visualisation)
    """
    tokens, attn = get_token_attention(text)
    words, weights = merge_wordpieces(tokens, attn)

    # Normalise weights 0-1
    if weights.max() > 0:
        weights = weights / weights.max()

    # Top attended words (filter out punctuation / stopwords)
    STOPWORDS = {'the','a','an','is','was','were','of','in','to','and',
                 'or','with','for','on','at','by','from','that','this',
                 'he','she','his','her','patient','pt','dr','mr','ms'}
    eligible = [(w, sc) for w, sc in zip(words, weights)
                if len(w) > 2 and w.lower() not in STOPWORDS]
    top_words = sorted(eligible, key=lambda x: x[1], reverse=True)[:top_n]

    # Categorise top words
    high_risk_signals  = []
    protective_signals = []
    neutral_signals    = []

    for word, score in top_words:
        category = classify_word_risk(word)
        plain    = plain_name(word)
        entry = {'word': word, 'plain': plain, 'attention': round(float(score), 3)}
        if category == 'high_risk':
            high_risk_signals.append(entry)
        elif category == 'moderate_risk':
            neutral_signals.append(entry)
        elif category == 'protective':
            protective_signals.append(entry)
        else:
            neutral_signals.append(entry)

    # Generate plain-language summary
    summary_parts = []
    if high_risk_signals:
        terms = ', '.join(s['plain'] for s in high_risk_signals[:3])
        summary_parts.append(f'The note highlights concerning findings: {terms}.')
    if protective_signals:
        terms = ', '.join(s['plain'] for s in protective_signals[:2])
        summary_parts.append(f'Some reassuring signs noted: {terms}.')
    if not summary_parts:
        summary_parts.append('No strongly risk-indicating clinical terms identified.')

    return {
        'summary':           ' '.join(summary_parts),
        'high_risk_signals': high_risk_signals,
        'protective_signals':protective_signals,
        'other_signals':     neutral_signals,
        'top_words':         top_words,
        'all_words':         words,
        'all_weights':       weights,
    }


# ── Demo on first available note
demo_note = notes_clean['text'].iloc[0]
result = explain_note_text(demo_note)

print('=' * 60)
print('CLINICAL NOTE EXPLANATION — PLAIN LANGUAGE')
print('=' * 60)
print(f'\nSUMMARY:\n  {result["summary"]}')

if result['high_risk_signals']:
    print('\nHIGH-RISK SIGNALS (model focused heavily on these):')
    for s in result['high_risk_signals']:
        bar = '#' * int(s['attention'] * 20)
        print(f'  [{bar:<20}] {s["attention"]:.3f}  "{s["word"]}" '
              f'= {s["plain"]}')

if result['protective_signals']:
    print('\nREASSURING SIGNALS (these reduce the predicted risk):')
    for s in result['protective_signals']:
        bar = '#' * int(s['attention'] * 20)
        print(f'  [{bar:<20}] {s["attention"]:.3f}  "{s["word"]}" '
              f'= {s["plain"]}')

print('\nOTHER ATTENDED TERMS:')
for s in result['other_signals'][:4]:
    print(f'  {s["attention"]:.3f}  "{s["word"]}" = {s["plain"]}')


### Visualise — Token Attention Heatmap
> Words highlighted in red = model paid high attention (risk signal).  
> Words in green = protective / reassuring signal.  
> Grey = low attention / neutral.


In [ ]:
def plot_text_attention(result: dict, title: str = 'Clinical Note Attention') -> None:
    """Render token attention as a colour-coded word cloud strip."""
    words   = result['all_words'][:60]   # first 60 words for readability
    weights = result['all_weights'][:60]

    fig, ax = plt.subplots(figsize=(20, 5), facecolor=DARK_BG)
    ax.set_facecolor(CARD_BG)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title(title, color=TEXT, fontsize=12, pad=10)

    # Lay out words in rows
    x, y = 0.01, 0.85
    line_height = 0.18
    char_width  = 0.013

    for word, weight in zip(words, weights):
        category = classify_word_risk(word)

        # Colour by category + intensity
        if category == 'high_risk':
            bg_color = (0.86, 0.15 + weight*0.2, 0.15 + weight*0.1)
            txt_color = 'white'
        elif category == 'protective':
            bg_color = (0.1, 0.5 + weight*0.3, 0.2)
            txt_color = 'white'
        elif category == 'moderate_risk':
            bg_color = (0.8, 0.5 + weight*0.2, 0.1)
            txt_color = 'white'
        else:
            intensity = 0.2 + weight * 0.5
            bg_color = (intensity, intensity, intensity+0.1)
            txt_color = 'white' if weight > 0.4 else '#94A3B8'

        word_w = len(word) * char_width + 0.015

        # Wrap to next line
        if x + word_w > 0.99:
            x = 0.01
            y -= line_height
            if y < 0.05:
                break

        # Draw word background box
        rect = plt.Rectangle((x, y - 0.08), word_w, 0.12,
                              facecolor=bg_color, edgecolor='none',
                              transform=ax.transAxes, clip_on=False)
        ax.add_patch(rect)

        ax.text(x + word_w/2, y - 0.02, word,
                color=txt_color, fontsize=8, fontweight='bold',
                ha='center', va='center', transform=ax.transAxes)

        x += word_w + 0.005

    # Legend
    legend_items = [
        mpatches.Patch(color='#DC2626', label='High-risk clinical signal'),
        mpatches.Patch(color='#EA580C', label='Moderate-risk signal'),
        mpatches.Patch(color='#16A34A', label='Protective / reassuring'),
        mpatches.Patch(color='#475569', label='Neutral / low attention'),
    ]
    ax.legend(handles=legend_items, loc='lower right',
              framealpha=0.3, facecolor=CARD_BG,
              labelcolor=TEXT, fontsize=8)

    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/Multimoal_ICUAIAssistant_Claud/'
                'icu_text_attention_demo.png',
                dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.show()


plot_text_attention(result, title='Clinical Note — Token Attention Heatmap')


## Explainability B — Grad-CAM for Chest X-Ray
> Grad-CAM highlights which regions of the CXR the DenseNet model focused on.  
> Warm colours (red/orange) = high attention region.  
> Each region is mapped to its anatomical name and a plain-English clinical meaning.


In [ ]:
import torch.nn.functional as F
from PIL import Image
import io

# ── Anatomical region map for CXR
# Divides a PA CXR into a 3x3 grid and names each zone
CXR_REGION_MAP = {
    (0,0): ('Upper left lung',  'Left upper lobe — check for pneumonia, mass'),
    (0,1): ('Upper central',    'Trachea / upper mediastinum — check for widening'),
    (0,2): ('Upper right lung', 'Right upper lobe — TB, pneumonia, mass'),
    (1,0): ('Left mid lung',    'Left hilum and mid zone — check for consolidation'),
    (1,1): ('Heart / mediastinum','Cardiac silhouette — enlarged = heart failure risk'),
    (1,2): ('Right mid lung',   'Right hilum and mid zone — consolidation, effusion'),
    (2,0): ('Left base / costophrenic','Left lower lobe and angle — fluid, atelectasis'),
    (2,1): ('Lower central',    'Lower mediastinum / diaphragm'),
    (2,2): ('Right base / costophrenic','Right lower lobe — most common pneumonia site'),
}

PATHOLOGY_PLAIN = {
    'Consolidation':       'areas of lung filled with fluid or pus (pneumonia pattern)',
    'Pleural Effusion':    'fluid around the lung',
    'Atelectasis':         'collapsed or partially collapsed lung area',
    'Pneumothorax':        'air leak collapsing the lung',
    'Cardiomegaly':        'enlarged heart shadow',
    'Edema':               'fluid in lung tissue (heart failure pattern)',
    'Lung Opacity':        'haziness in lung field — infection or fluid',
    'Fracture':            'rib or bone fracture visible',
    'Support Devices':     'tubes, lines, or devices placed in patient',
    'No Finding':          'no obvious abnormality detected',
    'Enlarged Cardiomediastinum': 'widened heart and central chest structures',
}


class GradCAMExtractor:
    """Registers forward/backward hooks on DenseNet's last dense block."""

    def __init__(self, model):
        self.model      = model
        self.gradients  = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        target_layer = self.model.features.denseblock4

        def forward_hook(module, inp, out):
            self.activations = out.detach()

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        target_layer.register_forward_hook(forward_hook)
        target_layer.register_full_backward_hook(backward_hook)

    def generate_cam(self, img_tensor: torch.Tensor,
                     target_class_idx: int) -> np.ndarray:
        """
        img_tensor : (1, 1, 224, 224)
        Returns cam : (224, 224) normalised heatmap [0, 1]
        """
        self.model.zero_grad()
        img_tensor.requires_grad_(True)

        logits = self.model(img_tensor)          # (1, n_labels)
        score  = logits[0, target_class_idx]
        score.backward()

        # Global average pool of gradients
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)  # (1,C,1,1)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # (1,1,H,W)
        cam = F.relu(cam).squeeze().cpu().numpy()

        # Resize to 224×224 and normalise
        cam = np.maximum(cam, 0)
        cam_resized = F.interpolate(
            torch.tensor(cam).unsqueeze(0).unsqueeze(0).float(),
            size=(224, 224), mode='bilinear', align_corners=False
        ).squeeze().numpy()

        if cam_resized.max() > 0:
            cam_resized /= cam_resized.max()
        return cam_resized


def get_top_cam_regions(cam: np.ndarray,
                        threshold: float = 0.5) -> list[dict]:
    """
    Divide 224x224 CAM into 3x3 grid, compute mean activation per cell.
    Returns top regions with anatomical names and clinical meanings.
    """
    h, w = cam.shape
    grid_h, grid_w = h // 3, w // 3
    regions = []

    for row in range(3):
        for col in range(3):
            cell = cam[row*grid_h:(row+1)*grid_h,
                       col*grid_w:(col+1)*grid_w]
            mean_act = float(cell.mean())
            if mean_act >= threshold:
                name, meaning = CXR_REGION_MAP.get((row, col),
                    ('Unknown region', 'Unknown region'))
                regions.append({
                    'region':   name,
                    'meaning':  meaning,
                    'activation': round(mean_act, 3),
                    'grid_pos': (row, col)
                })

    return sorted(regions, key=lambda x: x['activation'], reverse=True)


def explain_cxr(gcs_path: str,
                pathology_probs: dict,
                target_label: str = 'Consolidation') -> None:
    """
    Full Grad-CAM explanation for one CXR.
    Shows: original image | CAM overlay | plain-language panel
    """
    # ── Load and preprocess image
    img_arr = load_cxr_from_gcs(gcs_path)
    if img_arr is None:
        print(f'Could not load image: {gcs_path}')
        return

    transform = __import__('torchxrayvision').datasets
    img_proc  = xrv.datasets.XRayCenterCrop()(img_arr)
    img_proc  = xrv.datasets.XRayResizer(224)(img_proc)
    img_tensor = torch.tensor(img_proc, dtype=torch.float32)\
                      .unsqueeze(0).to(DEVICE)

    # ── Find target class index
    label_list = [str(l) for l in cxr_model.pathologies]
    if target_label not in label_list:
        target_label = label_list[0]
    target_idx = label_list.index(target_label)

    # ── Generate Grad-CAM
    grad_cam = GradCAMExtractor(cxr_model)
    cam = grad_cam.generate_cam(img_tensor, target_idx)

    # ── Top activated regions
    top_regions = get_top_cam_regions(cam, threshold=0.3)

    # ── Plot
    fig = plt.figure(figsize=(20, 8), facecolor=DARK_BG)
    gs_cxr = gridspec.GridSpec(1, 3, figure=fig,
                               width_ratios=[1, 1, 1.2], wspace=0.08)

    # Panel 1: Original CXR
    ax_orig = fig.add_subplot(gs_cxr[0])
    ax_orig.imshow(img_proc[0], cmap='gray', aspect='auto')
    ax_orig.set_title('Original CXR', color=TEXT, fontsize=11)
    ax_orig.axis('off')

    # Panel 2: CAM overlay
    ax_cam = fig.add_subplot(gs_cxr[1])
    ax_cam.imshow(img_proc[0], cmap='gray', aspect='auto')
    ax_cam.imshow(cam, cmap='jet', alpha=0.45, aspect='auto')
    ax_cam.set_title(f'Grad-CAM: model focus for "{target_label}"',
                     color=TEXT, fontsize=11)
    ax_cam.axis('off')

    # Grid overlay to show regions
    for r in top_regions[:3]:
        row, col = r['grid_pos']
        x0, y0 = col * 224/3, row * 224/3
        rect = plt.Rectangle((x0, y0), 224/3, 224/3,
                              fill=False, edgecolor='yellow',
                              linewidth=1.5, linestyle='--')
        ax_cam.add_patch(rect)
        ax_cam.text(x0 + 4, y0 + 14, f'{r["activation"]:.2f}',
                    color='yellow', fontsize=7, fontweight='bold')

    # Panel 3: Plain-language explanation
    ax_txt = fig.add_subplot(gs_cxr[2])
    ax_txt.set_facecolor(CARD_BG)
    ax_txt.axis('off')

    # Top pathology probabilities
    top_probs = sorted(pathology_probs.items(),
                       key=lambda x: x[1], reverse=True)[:6]

    lines = []
    lines.append(('WHAT THE MODEL DETECTED', '#F1F5F9', 13, True))
    lines.append(('', '', 9, False))

    for label, prob in top_probs:
        plain = PATHOLOGY_PLAIN.get(label, label)
        pct   = prob * 100
        color = '#DC2626' if pct>50 else '#EA580C' if pct>25 else '#94A3B8'
        lines.append((f'{label}: {pct:.0f}%', color, 10, True))
        lines.append((f'  = {plain}', '#94A3B8', 9, False))

    lines.append(('', '', 9, False))
    lines.append(('WHERE THE MODEL LOOKED', '#F1F5F9', 13, True))
    lines.append(('', '', 9, False))

    for r in top_regions[:4]:
        act_pct = r['activation'] * 100
        color = '#DC2626' if act_pct>60 else '#EA580C' if act_pct>40 else '#CA8A04'
        lines.append((f'{r["region"]} ({act_pct:.0f}% focus)', color, 10, True))
        lines.append((f'  {r["meaning"]}', '#94A3B8', 8, False))

    y_pos = 0.97
    for text, color, size, bold in lines:
        if text:
            ax_txt.text(0.03, y_pos, text,
                        color=color, fontsize=size,
                        fontweight='bold' if bold else 'normal',
                        transform=ax_txt.transAxes, va='top')
        y_pos -= 0.062 if size >= 10 else 0.05
        if y_pos < 0.02:
            break

    plt.savefig('/content/drive/MyDrive/Multimoal_ICUAIAssistant_Claud/'
                f'icu_cxr_gradcam_{target_label.replace(" ","_")}.png',
                dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.show()

    # ── Print plain-language summary
    print('\n' + '='*60)
    print('CHEST X-RAY EXPLANATION — PLAIN LANGUAGE')
    print('='*60)
    print(f'\nModel focused on: {target_label}')
    print(f'  = {PATHOLOGY_PLAIN.get(target_label, target_label)}')
    print('\nKey regions the model examined:')
    for r in top_regions[:4]:
        print(f'  [{r["activation"]*100:.0f}% attention] '
              f'{r["region"]}: {r["meaning"]}')
    print('\nTop detected conditions:')
    for label, prob in top_probs:
        plain = PATHOLOGY_PLAIN.get(label, label)
        flag = ' <-- HIGH' if prob > 0.5 else ''
        print(f'  {prob*100:5.1f}%  {label}: {plain}{flag}')


# ── Run on first available CXR in the dataset
if len(cxr_filtered) > 0:
    demo_row   = cxr_filtered.iloc[0]
    demo_path  = demo_row['path']
    demo_stay  = demo_row['stay_id']

    # Get pathology probabilities for this stay
    demo_cxr_row = cxr_emb_df[cxr_emb_df['stay_id'] == demo_stay]
    if len(demo_cxr_row) > 0:
        prob_dict = {
            col.replace('cxr_prob_','').replace('_',' '): float(demo_cxr_row[col].iloc[0])
            for col in PROB_COLS_CXR
        }
        # Find highest probability pathology for Grad-CAM target
        top_label = max(prob_dict, key=prob_dict.get)
        print(f'Generating Grad-CAM for stay {demo_stay}')
        print(f'Target label: {top_label} ({prob_dict[top_label]*100:.1f}%)')
        explain_cxr(demo_path, prob_dict, target_label=top_label)
else:
    print('No CXR available in filtered set — check temporal window.')


## Explainability C — Combined Patient Explanation Card
> Single function that takes a `stay_id` and generates a complete  
> doctor-facing explanation combining text signals + CXR findings.


In [ ]:
def explain_patient(stay_id: int) -> None:
    """
    Full multimodal explanation card for one ICU patient.
    Combines: note attention + CXR Grad-CAM + plain language summary.
    """
    print('\n' + '='*65)
    print(f'  PATIENT EXPLANATION CARD — Stay ID: {stay_id}')
    print('='*65)

    # ── 1. Static patient info
    pat = cohort[cohort['stay_id'] == stay_id]
    if len(pat) > 0:
        p = pat.iloc[0]
        print(f'\nPatient overview:')
        static_row = static[static['stay_id'] == stay_id].iloc[0]
        print(f'  Age          : {static_row["age_icu"]} years')
        print(f'  Gender       : {"Male" if static_row["gender"]=="M" else "Female"}')
        print(f'  Admission    : {"Elective" if static_row["is_elective"] else "Emergency"}')
        print(f'  Chronic dx   : {"Yes" if static_row["has_chronic"] else "None recorded"}')

    # ── 2. Risk scores
    scores_df = pd.read_parquet(BASE / 'icu_risk_scores.parquet')
    score_row = scores_df[scores_df['stay_id'] == stay_id]
    if len(score_row) > 0:
        sr = score_row.iloc[0]
        tier_color = {
            'LOW':'(stable)','MODERATE':'(watch)','HIGH':'(elevated)',
            'SEVERE':'(!! urgent)','CRITICAL':'(!!! critical)'
        }
        print(f'\nRisk assessment:')
        print(f'  Criticality  : {sr["criticality_tier"]} '
              f'{tier_color.get(sr["criticality_tier"],"")}')
        print(f'  Mortality    : {sr["apache2_pred_mortality"]*100:.1f}% '
              f'(APACHE II predicted)')
        print(f'  Sepsis alert : {"YES — act now" if sr["sepsis_alert"] else "No"}')
        print(f'  Organ failure: {"Yes" if sr["organ_dysfunction"] else "No"} '
              f'({int(sr["organ_failure_count"])} organ(s) affected)')

    # ── 3. Text explanation
    hadm_id = cohort.loc[cohort['stay_id']==stay_id, 'hadm_id']
    if len(hadm_id) > 0:
        hadm = int(hadm_id.iloc[0])
        stay_notes = notes_clean[notes_clean['hadm_id'] == hadm]
        if len(stay_notes) > 0:
            note_text = stay_notes['text'].iloc[0]
            text_exp = explain_note_text(note_text)
            print(f'\nClinical note analysis ({len(stay_notes)} note(s)):')
            print(f'  {text_exp["summary"]}')
            if text_exp['high_risk_signals']:
                terms = ', '.join(
                    f"{s[\"plain\"]}" for s in text_exp['high_risk_signals'][:4]
                )
                print(f'  Risk signals : {terms}')
            if text_exp['protective_signals']:
                terms = ', '.join(
                    f"{s[\"plain\"]}" for s in text_exp['protective_signals'][:3]
                )
                print(f'  Reassuring   : {terms}')
        else:
            print('\nClinical note: not available for this stay')

    # ── 4. CXR explanation
    cxr_row = cxr_filtered[cxr_filtered['stay_id'] == stay_id]
    if len(cxr_row) > 0:
        cxr_emb_row = cxr_emb_df[cxr_emb_df['stay_id'] == stay_id]
        if len(cxr_emb_row) > 0:
            prob_dict = {
                col.replace('cxr_prob_','').replace('_',' '): float(cxr_emb_row[col].iloc[0])
                for col in PROB_COLS_CXR
            }
            top_findings = [(k,v) for k,v in
                            sorted(prob_dict.items(), key=lambda x: x[1], reverse=True)
                            if v > 0.2][:4]
            print(f'\nChest X-ray findings:')
            for label, prob in top_findings:
                plain = PATHOLOGY_PLAIN.get(label, label)
                flag = ' <-- significant' if prob > 0.5 else ''
                print(f'  {prob*100:5.1f}%  {label}: {plain}{flag}')
    else:
        print('\nChest X-ray: not available within admission window')

    print('\n' + '='*65)


# ── Demo: explain first patient in test set
test_stays = cohort[cohort['split']=='test']['stay_id'].values
if len(test_stays) > 0:
    explain_patient(test_stays[0])
    explain_patient(test_stays[1])   # second patient for comparison


## Step 12 — Output Summary & Next Steps

In [ ]:
print('=== MULTIMODAL ENCODER OUTPUTS ===')
for path in [TEXT_EMB_OUT, CXR_EMB_OUT, TEXT_RPT_PNG, CXR_RPT_PNG,
             MODEL_DIR / 'embedding_meta.json']:
    p = Path(path)
    exists = p.exists()
    size = p.stat().st_size / 1e6 if exists else 0
    print(f'  {"OK" if exists else "MISSING":6s} | {size:6.1f} MB | {p.name}')

print(f'\n=== COVERAGE SUMMARY ===')
print(f'  Total stays        : {len(cohort):,}')
print(f'  With text note     : {stay_text_emb["text_available"].sum():,}'
      f' ({stay_text_emb["text_available"].mean()*100:.1f}%)')
print(f'  With CXR image     : {cxr_full["cxr_available"].sum():,}'
      f' ({cxr_full["cxr_available"].mean()*100:.1f}%)')
print(f'  With both modalities: '
      f'{(stay_text_emb["text_available"].values & cxr_full["cxr_available"].values).sum():,}')

print(f'\n=== EMBEDDING DIMENSIONS ===')
print(f'  ClinicalBERT text embedding : 768 dims per stay')
print(f'  DenseNet CXR feature vector : {len(FEAT_COLS_CXR)} dims per stay')
print(f'  DenseNet CXR label probs    : {len(PROB_COLS_CXR)} pathologies per stay')
print(f'  Time-series XGBoost (prev.) : ~110 features per stay')
print(f'\n=== READY FOR FUSION LAYER ===')
print('  Next notebook: ICUFusionLayer.ipynb')
print('  Inputs: icu_encoder_predictions.parquet')
print('         icu_text_embeddings.parquet')
print('         icu_cxr_embeddings.parquet')
